In [3]:
def define_region_masks(pairs_df):
    n = len(pairs_df)
    masks = {}

    # Find best region column
    region_col = None
    for col_candidate in ['mni.region', 'ind.region', 'stein.region', 'avg.region', 'dk.region']:
        if col_candidate in pairs_df.columns:
            region_col = col_candidate
            break
    if region_col is None:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        region_col = region_cols[0] if region_cols else None

    if region_col is None:
        print("    ERROR: No region column found!")
        for r in REGIONS:
            masks[r] = np.zeros(n, dtype=bool)
        return masks

    regions_str = pairs_df[region_col].fillna('')

    # Hemisphere helper: prefer label prefix (L/R), fall back to region string
    def is_left(row_idx):
        if 'label' in pairs_df.columns:
            lbl = str(pairs_df['label'].iloc[row_idx])
            if lbl and lbl[0].upper() == 'L':
                return True
            if lbl and lbl[0].upper() == 'R':
                return False
        return 'left' in regions_str.iloc[row_idx].lower()

    left_mask  = np.array([is_left(i)     for i in range(n)])
    right_mask = np.array([not is_left(i) for i in range(n)])

    hipp_mask = regions_str.str.contains(
        'Hippocampus', case=False, na=False
    ).values
    ent_mask = regions_str.str.contains(
        'Entorhinal', case=False, na=False
    ).values
    par_mask = regions_str.str.contains(
        'Parietal', case=False, na=False
    ).values
    temporal_mask = regions_str.str.contains(
        r'Superior Temporal|Middle Temporal|Inferior Temporal|STG|MTG|ITG',
        case=False, na=False, regex=True
    ).values

    masks['LHP']  = hipp_mask    & left_mask
    masks['RHP']  = hipp_mask    & right_mask
    masks['LEC']  = ent_mask     & left_mask
    masks['REC']  = ent_mask     & right_mask
    masks['LPC']  = par_mask     & left_mask
    masks['RPC']  = par_mask     & right_mask
    masks['LLTC'] = temporal_mask & left_mask

    return masks

In [ ]:
#!/usr/bin/env python3
"""
IRASA Pipeline — Channel-level wide format, parallel execution
--------------------------------------------------------------
One row per recalled_word × channel

Experiments : DBOY1, EFRCourierReadOnly, EFRCourierOpenLoop
Regions     : LLTC, RLTC  (Left/Right Lateral Temporal Cortex: STG, MTG, ITG)

Output columns:
  metadata   : subject, session, experiment, trial, recalled_word,
               serial_position, recall_output_position, store_x, store_z,
               region, channel_label, channel_idx
  clustering : SP_clustering_to_next, T_clustering_to_next
  stimulation: recall_stim, encoding_stim
  encoding   : enc_frac_f00..f17, enc_osc_f00..f17  (18 cols each)
  retrieval  : ret_frac_f00..f17, ret_osc_f00..f17  (18 cols each)

Parallelism : session-level (each session is an independent worker)
Resume      : skips sessions whose CSV already exists
Output      : one CSV per session saved immediately after completion,
              one master CSV per experiment written at the end
"""

import os
import copy
import traceback
import warnings
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# IRASA IMPORTS
# ============================================================================
try:
    from irasa.IRASA import IRASA, SSL_transform
    IRASA_AVAILABLE = True
    print("✓ IRASA imported successfully")
except ImportError as e:
    IRASA_AVAILABLE = False
    print(f"✗ IRASA not available: {e}")
    def SSL_transform(x):
        return np.sign(x) * np.log10(1 + np.abs(x))

import cmlreaders as cml
print("✓ cmlreaders imported successfully")

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop']

IRASA_HSET  = np.arange(1.1, 1.9, 0.05)
IRASA_FREQS = np.round(np.logspace(np.log10(3), np.log10(128), 18), 2)
N_FREQS     = len(IRASA_FREQS)

REGIONS = ['LLTC', 'RLTC']

# Encoding window
ENCODING_EPOCH_START  = -2000
ENCODING_EPOCH_END    =  4000
ENCODING_WIN_START_MS =     0
ENCODING_WIN_END_MS   =   750

# Retrieval window
RETRIEVAL_EPOCH_START  = -4000
RETRIEVAL_EPOCH_END    =  2000
RETRIEVAL_WIN_START_MS =  -750
RETRIEVAL_WIN_END_MS   =     0

N_WORKERS = cpu_count()

# ============================================================================
# COLUMN NAMES
# ============================================================================

ENC_FRAC_COLS = [f'enc_frac_f{i:02d}' for i in range(N_FREQS)]
ENC_OSC_COLS  = [f'enc_osc_f{i:02d}'  for i in range(N_FREQS)]
RET_FRAC_COLS = [f'ret_frac_f{i:02d}' for i in range(N_FREQS)]
RET_OSC_COLS  = [f'ret_osc_f{i:02d}'  for i in range(N_FREQS)]
FREQ_COLS     = ENC_FRAC_COLS + ENC_OSC_COLS + RET_FRAC_COLS + RET_OSC_COLS

META_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'recalled_word', 'serial_position', 'recall_output_position',
    'store_x', 'store_z',
    'region', 'channel_label', 'channel_idx',
    'SP_clustering_to_next', 'T_clustering_to_next',
    'recall_stim', 'encoding_stim',
]
ALL_COLS = META_COLS + FREQ_COLS

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def ssl(x):
    return SSL_transform(x)

def ms_to_sample(ms, epoch_start_ms, sfreq):
    return int(round((ms - epoch_start_ms) / 1000.0 * sfreq))

def compute_euclidean_distance(c1, c2):
    return euclidean(c1, c2)

def build_spatial_distance_matrix(coords):
    n = len(coords)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = compute_euclidean_distance(coords[i], coords[j])
    return D

def build_temporal_distance_matrix(n_items):
    idx = np.arange(n_items)
    return np.abs(idx[:, None] - idx[None, :]).astype(float)

def convert_recalled_coords_to_itemnos(presented_coords, recalled_coords, tolerance=0.01):
    itemnos = []
    for rc in recalled_coords:
        dists = np.array([compute_euclidean_distance(rc, pc) for pc in presented_coords])
        ci    = np.argmin(dists)
        itemnos.append(ci + 1 if dists[ci] <= tolerance else -1)
    return np.array(itemnos)

def make_recalls_matrix(pres_itemnos, rec_itemnos):
    n_trials  = pres_itemnos.shape[0]
    n_recalls = rec_itemnos.shape[1]
    recalls   = np.zeros([n_trials, n_recalls], dtype=int)
    for t in range(n_trials):
        for r in range(n_recalls):
            v = rec_itemnos[t, r]
            if v == 0 or np.isnan(v):
                continue
            elif v > 0:
                sp = np.where(v == pres_itemnos[t, :])[0] + 1
                recalls[t, r] = sp[0] if len(sp) == 1 else -1
            else:
                recalls[t, r] = -1
    return recalls

def make_clean_recalls_mask2d(data):
    result = copy.deepcopy(data)
    for num, item in enumerate(data):
        seen = []
        for index, recall in enumerate(item):
            if recall > 0 and recall not in seen:
                result[num][index] = 1
                seen.append(recall)
            else:
                result[num][index] = 0
    return result

def dist_percentile_rank(actual, possible):
    if len(possible) < 2:
        return None
    possible_sorted = sorted(possible)[::-1]
    matches = np.where(np.array(possible_sorted) == actual)[0]
    if len(matches) == 0:
        return None
    return np.mean(matches) / (len(possible_sorted) - 1.)

def dist_fact_single_trial(rec_itemnos, pres_itemnos, dist_mat, skip_first_n=0):
    """
    Returns clustering score (percentile rank) for each transition → to_next.
    Output length == number of clean recalls. Last element is always NaN.
    """
    rec_itemnos  = np.array(rec_itemnos,  dtype=float)
    pres_itemnos = np.array(pres_itemnos, dtype=int)
    dist_mat     = np.array(dist_mat)

    if rec_itemnos.ndim  == 2: rec_itemnos  = rec_itemnos[0]
    if pres_itemnos.ndim == 2: pres_itemnos = pres_itemnos[0]

    rec_2d  = rec_itemnos.reshape(1, -1)
    pres_2d = pres_itemnos.reshape(1, -1)
    clean_mask = make_clean_recalls_mask2d(make_recalls_matrix(pres_2d, rec_2d))[0]

    scores = []
    seen   = set()

    for j, rec in enumerate(rec_itemnos[:-1]):
        rec = int(rec)
        seen.add(rec)
        if clean_mask[j] and clean_mask[j + 1] and j >= skip_first_n:
            possibles = np.array([
                dist_mat[rec - 1, int(p) - 1]
                for p in pres_itemnos if int(p) not in seen
            ])
            next_rec = int(rec_itemnos[j + 1])
            actual   = dist_mat[rec - 1, next_rec - 1]
            ptile    = dist_percentile_rank(actual, possibles)
            scores.append(ptile if ptile is not None else np.nan)
        else:
            scores.append(np.nan)

    scores.append(np.nan)   # last word has no to_next
    return np.array(scores)

def compute_clustering_scores(presented_coords, clean_rec_itemnos):
    n_items     = len(presented_coords)
    spatial_dm  = build_spatial_distance_matrix(presented_coords)
    temporal_dm = build_temporal_distance_matrix(n_items)

    pres_itemnos = np.array([list(range(1, n_items + 1))])
    rec_itemnos  = np.zeros((1, n_items), dtype=int)
    rec_itemnos[0, :len(clean_rec_itemnos)] = clean_rec_itemnos

    sp_scores = dist_fact_single_trial(rec_itemnos, pres_itemnos, spatial_dm)
    t_scores  = dist_fact_single_trial(rec_itemnos, pres_itemnos, temporal_dm)
    return sp_scores, t_scores

def add_phase_column(evs):
    evs = evs.copy()
    evs["phase"] = 1
    for ind, list_evs in evs.groupby('trial'):
        list_evs = list_evs.copy()
        if list_evs['trial'].unique()[0] != -999:
            list_evs['phase'] = 'nan'
            if (not list_evs[list_evs['type'] == 'TRIAL_START'].index.empty and
                    not list_evs[list_evs['type'] == 'TRIAL_END'].index.empty):
                rec_start_indices = list_evs[list_evs['type'] == 'REC_START'].index
                if not rec_start_indices.empty:
                    rec_start_index = rec_start_indices[0]
                    start_time = list_evs.loc[rec_start_index]['eegoffset']
                    i = 0
                    found = True
                    actual_start = rec_start_index
                    while found:
                        current_idx = rec_start_index + i
                        check_time  = evs.iloc[current_idx]['eegoffset']
                        if check_time - start_time < 3000:
                            i += 1
                        else:
                            found = False
                            actual_start = rec_start_index + i
                    rec_end_indices = list_evs[list_evs['type'] == 'REC_STOP'].index
                    if not rec_end_indices.empty:
                        rec_end_index = rec_end_indices[0]
                        evs.loc[actual_start:rec_end_index, 'phase'] = 'retrieval'
    return evs

def mark_stim_events(evs):
    evs = evs.copy()
    evs['inside_stimuli'] = -999
    for i in evs[evs['type'] == 'STIM'].index:
        current_offset = evs.loc[i, 'eegoffset']
        burst_freq     = evs.loc[i, 'stim_params']['burst_freq']
        evs.loc[i, 'inside_stimuli'] = burst_freq
        j = i + 1
        while j < len(evs):
            if abs(evs.loc[j, 'eegoffset'] - current_offset) < 3000:
                evs.loc[j, 'inside_stimuli'] = burst_freq
                j += 1
            else:
                break
    word_indices = evs[evs['type'] == 'WORD'].index
    preceding    = evs.iloc[word_indices - 1]
    for stim_idx in preceding[preceding['type'] == 'STIM'].index:
        word_idx   = stim_idx + 1
        burst_freq = evs.loc[stim_idx, 'stim_params']['burst_freq']
        evs.loc[word_idx, 'inside_stimuli'] = burst_freq
    return evs

def define_region_masks(pairs_df):
    """
    Boolean masks for Left and Right Lateral Temporal Cortex.
    LLTC / RLTC = STG, MTG, ITG on left / right hemisphere.

    Hemisphere is inferred first from the channel label prefix (L/R),
    then falls back to the region string containing 'left'/'right'.
    """
    n = len(pairs_df)
    masks = {}

    # ── Find best region column ───────────────────────────────────────────────
    region_col = None
    for col_candidate in ['mni.region', 'ind.region', 'stein.region',
                          'avg.region', 'dk.region']:
        if col_candidate in pairs_df.columns:
            region_col = col_candidate
            break
    if region_col is None:
        region_cols = [c for c in pairs_df.columns if 'region' in c.lower()]
        region_col  = region_cols[0] if region_cols else None

    if region_col is None:
        print("    ERROR: No region column found!")
        for r in REGIONS:
            masks[r] = np.zeros(n, dtype=bool)
        return masks

    print(f"    Using region column: {region_col}")
    regions_str = pairs_df[region_col].fillna('')

    # ── Hemisphere helper: label prefix wins, falls back to region string ─────
    def is_left(row_idx):
        if 'label' in pairs_df.columns:
            lbl = str(pairs_df['label'].iloc[row_idx])
            if lbl and lbl[0].upper() == 'L':
                return True
            if lbl and lbl[0].upper() == 'R':
                return False
        return 'left' in regions_str.iloc[row_idx].lower()

    left_mask  = np.array([is_left(i)     for i in range(n)])
    right_mask = ~left_mask

    # ── Lateral temporal cortex: STG / MTG / ITG ─────────────────────────────
    ltc_mask = regions_str.str.contains(
        r'Superior Temporal|Middle Temporal|Inferior Temporal|STG|MTG|ITG',
        case=False, na=False, regex=True
    ).values

    masks['LLTC'] = ltc_mask & left_mask
    masks['RLTC'] = ltc_mask & right_mask

    return masks

def compute_irasa_channel(sig, sfreq):
    """Run IRASA on a single 1-D signal. Returns (frac_ssl, osc_ssl) of length N_FREQS."""
    ir   = IRASA(sig=sig.astype(float), freqs=IRASA_FREQS, samplerate=sfreq,
                 hset=IRASA_HSET, flag_filter=1, flag_detrend=1)
    frac = ir.fractal
    osc  = ir.mixed - frac
    return ssl(frac), ssl(osc)

# ============================================================================
# PER-SESSION WORKER
# ============================================================================

def process_session(args):
    """
    Process one (subject, session, experiment, output_dir) tuple.
    Saves a CSV immediately on completion.

    Returns
    -------
    str  – log messages
    int  – number of rows written
    """
    subject, session, experiment, output_dir = args
    msgs = []
    rows = []

    tag   = f"[{subject} sess={session} exp={experiment}]"
    fpath = Path(output_dir) / f"{subject}_{session}_{experiment}_irasa_LTC_wide.csv"

    # Resume support
    if fpath.exists():
        msgs.append(f"   {tag} Already exists – skipping")
        return '\n'.join(msgs), 0

    try:
        reader       = cml.CMLReader(subject, experiment, session=session)
        pairs_df     = reader.load('pairs')
        region_masks = define_region_masks(pairs_df)

        region_summary = {r: int(m.sum()) for r, m in region_masks.items()}
        msgs.append(f"   {tag} pairs={len(pairs_df)} | regions={region_summary}")

        # Build channel list: (region, ch_idx, channel_label)
        channel_info = []
        for region in REGIONS:
            for ch_idx in np.where(region_masks[region])[0]:
                row_p = pairs_df.iloc[int(ch_idx)]
                label = str(row_p.get('label', row_p.get('tagName', f'ch{ch_idx}')))
                channel_info.append((region, int(ch_idx), label))

        if len(channel_info) == 0:
            msgs.append(f"   {tag} No LLTC/RLTC channels found – skipping")
            return '\n'.join(msgs), 0

        unique_ch_indices = list({ci for _, ci, _ in channel_info})

        evs = reader.load('task_events')
        evs = add_phase_column(evs)
        evs = mark_stim_events(evs)

        for trial in sorted(evs['trial'].unique()):
            if trial == -999 or trial < 0:
                continue

            trial_evs         = evs[evs['trial'] == trial]
            trial_evs_RECWORD = trial_evs[trial_evs['type'] == 'REC_WORD']
            trial_evs_WORD    = trial_evs[trial_evs['type'] == 'WORD']

            if len(trial_evs_RECWORD) == 0 or len(trial_evs_WORD) == 0:
                continue

            try:
                # ── Coordinates & clean recalls ──────────────────────────────
                recalled_coords  = np.column_stack([trial_evs_RECWORD['storeX'],
                                                    trial_evs_RECWORD['storeZ']])
                presented_coords = np.column_stack([trial_evs_WORD['storeX'],
                                                    trial_evs_WORD['storeZ']])
                rec_itemnos_array = convert_recalled_coords_to_itemnos(
                    presented_coords, recalled_coords
                )

                pres_2d = np.array([list(range(1, len(presented_coords) + 1))])
                rec_2d  = np.zeros((1, len(presented_coords)), dtype=int)
                rec_2d[0, :len(rec_itemnos_array)] = rec_itemnos_array

                clean_mask        = make_clean_recalls_mask2d(
                    make_recalls_matrix(pres_2d, rec_2d))[0]
                clean_recall_idx  = np.where(clean_mask == 1)[0]
                clean_rec_itemnos = rec_itemnos_array[clean_recall_idx]

                if len(clean_rec_itemnos) == 0:
                    continue

                # ── Clustering scores ─────────────────────────────────────────
                sp_scores, t_scores = compute_clustering_scores(
                    presented_coords, clean_rec_itemnos
                )

                # ── Load & trim encoding EEG ──────────────────────────────────
                eeg_enc_cont = reader.load_eeg(
                    trial_evs_WORD,
                    ENCODING_EPOCH_START, ENCODING_EPOCH_END,
                    scheme=pairs_df
                )
                eeg_enc  = eeg_enc_cont.data
                sfreq    = float(eeg_enc_cont.samplerate)
                s0e      = ms_to_sample(ENCODING_WIN_START_MS, ENCODING_EPOCH_START, sfreq)
                s1e      = ms_to_sample(ENCODING_WIN_END_MS,   ENCODING_EPOCH_START, sfreq)
                eeg_enc_win = eeg_enc[:, :, s0e:s1e]   # (n_words, n_ch, samples)

                # ── Load & trim retrieval EEG ─────────────────────────────────
                clean_RECWORD_evs = trial_evs_RECWORD.iloc[clean_recall_idx].reset_index(drop=True)
                eeg_ret_cont = reader.load_eeg(
                    clean_RECWORD_evs,
                    RETRIEVAL_EPOCH_START, RETRIEVAL_EPOCH_END,
                    scheme=pairs_df
                )
                eeg_ret = eeg_ret_cont.data
                s0r     = ms_to_sample(RETRIEVAL_WIN_START_MS, RETRIEVAL_EPOCH_START, sfreq)
                s1r     = ms_to_sample(RETRIEVAL_WIN_END_MS,   RETRIEVAL_EPOCH_START, sfreq)
                eeg_ret_win = eeg_ret[:, :, s0r:s1r]   # (n_recalls, n_ch, samples)

                # ── Pre-compute IRASA per channel (encoding) ──────────────────
                n_words   = eeg_enc_win.shape[0]
                enc_irasa = {}   # ch_idx → {'frac': (n_words, N_FREQS), 'osc': ...}
                for ch_idx in unique_ch_indices:
                    frac_rows, osc_rows = [], []
                    for word_idx in range(n_words):
                        sig = eeg_enc_win[word_idx, ch_idx, :]
                        try:
                            f, o = compute_irasa_channel(sig, sfreq)
                        except Exception:
                            f = np.full(N_FREQS, np.nan)
                            o = np.full(N_FREQS, np.nan)
                        frac_rows.append(f)
                        osc_rows.append(o)
                    enc_irasa[ch_idx] = {
                        'frac': np.array(frac_rows),
                        'osc':  np.array(osc_rows),
                    }

                # ── Pre-compute IRASA per channel (retrieval) ─────────────────
                n_recalls = eeg_ret_win.shape[0]
                ret_irasa = {}   # ch_idx → {'frac': (n_recalls, N_FREQS), 'osc': ...}
                for ch_idx in unique_ch_indices:
                    frac_rows, osc_rows = [], []
                    for recall_idx in range(n_recalls):
                        sig = eeg_ret_win[recall_idx, ch_idx, :]
                        try:
                            f, o = compute_irasa_channel(sig, sfreq)
                        except Exception:
                            f = np.full(N_FREQS, np.nan)
                            o = np.full(N_FREQS, np.nan)
                        frac_rows.append(f)
                        osc_rows.append(o)
                    ret_irasa[ch_idx] = {
                        'frac': np.array(frac_rows),
                        'osc':  np.array(osc_rows),
                    }

                # ── Build rows ────────────────────────────────────────────────
                for clean_idx, rec_itemno in enumerate(clean_rec_itemnos):
                    rec_word_row  = clean_RECWORD_evs.iloc[clean_idx]
                    pres_word_row = trial_evs_WORD.iloc[int(rec_itemno) - 1]

                    recall_stim   = rec_word_row['inside_stimuli']
                    encoding_stim = pres_word_row['inside_stimuli']

                    for region, ch_idx, ch_label in channel_info:
                        row = {
                            'subject':                subject,
                            'session':                session,
                            'experiment':             experiment,
                            'trial':                  trial,
                            'recalled_word':          rec_word_row['item'],
                            'serial_position':        int(rec_itemno),
                            'recall_output_position': int(clean_idx + 1),
                            'store_x':                rec_word_row['storeX'],
                            'store_z':                rec_word_row['storeZ'],
                            'region':                 region,
                            'channel_label':          ch_label,
                            'channel_idx':            ch_idx,
                            'SP_clustering_to_next':  sp_scores[clean_idx],
                            'T_clustering_to_next':   t_scores[clean_idx],
                            'recall_stim':            recall_stim,
                            'encoding_stim':          encoding_stim,
                        }

                        # Encoding IRASA — index by serial position
                        word_enc_idx = int(rec_itemno) - 1
                        frac_enc = enc_irasa[ch_idx]['frac'][word_enc_idx]
                        osc_enc  = enc_irasa[ch_idx]['osc'][word_enc_idx]
                        for fi in range(N_FREQS):
                            row[f'enc_frac_f{fi:02d}'] = frac_enc[fi]
                            row[f'enc_osc_f{fi:02d}']  = osc_enc[fi]

                        # Retrieval IRASA — index by recall output position
                        frac_ret = ret_irasa[ch_idx]['frac'][clean_idx]
                        osc_ret  = ret_irasa[ch_idx]['osc'][clean_idx]
                        for fi in range(N_FREQS):
                            row[f'ret_frac_f{fi:02d}'] = frac_ret[fi]
                            row[f'ret_osc_f{fi:02d}']  = osc_ret[fi]

                        rows.append(row)

            except Exception as e:
                msgs.append(f"   {tag} Trial {trial} FAILED: {e}")
                msgs.append(traceback.format_exc())
                continue

        # ── Save session CSV immediately ──────────────────────────────────────
        if rows:
            df = pd.DataFrame(rows, columns=ALL_COLS)
            df.to_csv(fpath, index=False)
            msgs.append(
                f"   ✓ {tag} Saved {len(rows):,} rows → {fpath.name} "
                f"({df['channel_label'].nunique()} channels, "
                f"{df['trial'].nunique()} trials)"
            )
        else:
            msgs.append(f"   {tag} No rows generated – CSV not written")

    except Exception as e:
        msgs.append(f"   {tag} FAILED: {e}")
        msgs.append(traceback.format_exc())
        return '\n'.join(msgs), 0

    return '\n'.join(msgs), len(rows)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':

    whole_df = cml.CMLReader.get_data_index()

    for exp in EXPERIMENTS:
        print(f"\n{'='*80}")
        print(f"EXPERIMENT: {exp}")
        print(f"{'='*80}")

        output_dir = Path(f'./subject_results_LTC_{exp}')
        output_dir.mkdir(exist_ok=True)

        exp_df   = whole_df[whole_df['experiment'] == exp]
        subjects = sorted(exp_df['subject'].unique())
        print(f"Found {len(subjects)} subjects")

        # Build full job list: one job per session
        job_args = []
        for subject in subjects:
            for session in exp_df[exp_df['subject'] == subject]['session'].unique():
                job_args.append((subject, int(session), exp, str(output_dir)))

        # Filter out already-completed sessions
        pending = [
            a for a in job_args
            if not (output_dir / f"{a[0]}_{a[1]}_{exp}_irasa_LTC_wide.csv").exists()
        ]

        print(f"Total sessions      : {len(job_args)}")
        print(f"Already done        : {len(job_args) - len(pending)}")
        print(f"Sessions to process : {len(pending)}")
        print(f"Launching {N_WORKERS} parallel workers …\n" + "="*80)

        if pending:
            with Pool(processes=N_WORKERS) as pool:
                results = pool.map(process_session, pending)

            # Print all logs
            print(f"\n{'='*80}")
            print(f"SESSION RESULTS — {exp}")
            print(f"{'='*80}")
            total_rows = 0
            for msg, n_rows in results:
                print(msg)
                total_rows += n_rows
            print(f"\nTotal rows written this run: {total_rows:,}")
        else:
            print("All sessions already processed.")

        # Concatenate all session CSVs into master
        print(f"\n{'='*80}")
        print(f"BUILDING MASTER CSV — {exp}")
        print(f"{'='*80}")

        all_files = sorted(output_dir.glob(f"*_{exp}_irasa_LTC_wide.csv"))
        if all_files:
            master_df  = pd.concat(
                [pd.read_csv(f) for f in all_files],
                ignore_index=True
            )
            master_path = output_dir / f"ALL_SUBJECTS_{exp}_irasa_LTC_wide.csv"
            master_df.to_csv(master_path, index=False)
            print(f"  Subjects : {master_df['subject'].nunique()}")
            print(f"  Sessions : {master_df['session'].nunique()}")
            print(f"  Channels : {master_df['channel_label'].nunique()}")
            print(f"  Regions  : {sorted(master_df['region'].unique())}")
            print(f"  Rows     : {len(master_df):,}")
            print(f"  → {master_path}")
        else:
            print("  No session CSVs found – master not written")

    print(f"\n{'='*80}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*80}")

✓ IRASA imported successfully
✓ cmlreaders imported successfully

EXPERIMENT: DBOY1
Found 44 subjects
Total sessions      : 91
Already done        : 0
Sessions to process : 91
Launching 40 parallel workers …
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
    Using region column: mni.region
